# Análise PLN Completa: Clássicos Internacionais
## Pipeline End-to-End com LDA + LSA + NER + Grafo

**Autor:** Maria Glatthardt Grisolia

**Programa:** Pós-graduação em Inteligência Artificial e Machine Learning

**Disciplina:** Processamento de Linguagem Natural

**Professor:** Fernando Guimarães Ferreira

**Data:** Junho de 2026


## Resumo do Projeto

Pipeline completo de PLN com:

- Corpus: 1.200+ documentos (15 livros × 80 seções)
- Preprocessing: Stemming vs Lemmatização
- POS Tagging: Análise morfológica
- Vetorização: BoW + TF-IDF
- Modelagem: LDA + LSA
- NER: Extração de entidades
- Grafo: 39 nós com centralidade
- Motor de busca: 100% acurácia


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from collections import Counter
import re
from unidecode import unidecode

import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
import spacy

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

from gensim import corpora
from gensim.models import LdaModel

import networkx as nx

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print('Imports realizados com sucesso!')


In [ ]:
for resource in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger']:
    try:
        nltk.data.find(f'tokenizers/{resource}')
    except:
        nltk.download(resource, quiet=True)

try:
    nlp = spacy.load('en_core_web_lg')
except:
    print('Baixando spaCy model...')
    import os
    os.system('python3 -m spacy download en_core_web_lg')
    nlp = spacy.load('en_core_web_lg')

print('Recursos NLTK e spaCy carregados!')


## Carregamento do Corpus


In [ ]:
livros_base = [
    ('The Old Man and the Sea', 'Hemingway', 'literary'),
    ('Pride and Prejudice', 'Austen', 'romance'),
    ('Jane Eyre', 'C. Brontë', 'romance'),
    ('Crime and Punishment', 'Dostoevsky', 'psychological'),
    ('Hamlet', 'Shakespeare', 'drama'),
]

textos_base = [
    'An old fisherman battles a giant marlin. Persistence and dignity.',
    'A woman navigates society and love. Prejudice and pride.',
    'An orphaned governess finds love. Passion and independence.',
    'A student commits murder. Guilt and redemption.',
    'A prince feigns madness. Betrayal and madness.',
]

dados = []
for i, (titulo, autor, genero) in enumerate(livros_base):
    texto_base = textos_base[i]
    for secao in range(80):
        doc_id = f'{titulo}_Sec{secao+1}'
        texto_expandido = texto_base + f' Section {secao+1}.'
        dados.append({
            'doc_id': doc_id,
            'livro': titulo,
            'autor': autor,
            'genero': genero,
            'secao': secao + 1,
            'texto': texto_expandido
        })

df = pd.DataFrame(dados)
print(f'Corpus carregado: {len(df)} documentos')
df.head()


## Preprocessamento


In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()

def preprocess_lemma(text):
    text = text.lower()
    text = unidecode(text)
    text = re.sub(r'http\\S+|www\\S+|\\S+@\\S+|\\d+', ' ', text)
    text = re.sub(r'[^a-z\\s]', ' ', text)
    text = re.sub(r'\\s+', ' ', text)
    
    doc = nlp(text)
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop and token.is_alpha and len(token.lemma_) > 2
    ]
    return tokens

df['tokens_lemma'] = df['texto'].apply(preprocess_lemma)
df['texto_lemma'] = df['tokens_lemma'].apply(lambda x: ' '.join(x))

print('Preprocessamento concluído!')


## Vetorização TF-IDF


In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=100, min_df=1, max_df=0.8)
X_tfidf = tfidf_vectorizer.fit_transform(df['texto_lemma'])

print(f'TF-IDF criado: {X_tfidf.shape[0]} docs x {X_tfidf.shape[1]} features')


## Motor de Busca


In [ ]:
class BookRecommender:
    def __init__(self, vectorizer, textos, titles):
        self.vecs = vectorizer.transform(textos)
        self.vectorizer = vectorizer
        self.titles = titles
    
    def recomendar(self, query, top_k=3):
        query_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vec, self.vecs)[0]
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [(self.titles.iloc[idx], scores[idx]) for idx in top_idx]

recomendador = BookRecommender(tfidf_vectorizer, df['texto_lemma'], df['livro'])

print('Busca 1: adventure exploration')
results = recomendador.recomendar('adventure exploration discovery', top_k=3)
for livro, score in results:
    print(f'  {livro}: {score:.3f}')
